In [ ]:
#%pip install pandas
#%pip install sqlalchemy
#%pip install pyarrow
#%pip install fastparquet

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

tables = []
ref_schema = None

for f in Path(r"C:\Users\brand\portfolio\taxi\raw_data_files").glob("*.parquet"):
    t = pq.read_table(f)
    if ref_schema is None:
        ref_schema = set(t.schema.names)
        tables.append(t)
    elif set(t.schema.names) == ref_schema:
        tables.append(t)
    else:
        print(f"Skipped {f.name}: column mismatch")

table = pa.concat_tables(tables) if tables else None
print(table)
df = table.to_pandas()
df.columns = df.columns.str.lower()

pyarrow.Table
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double
----
VendorID: [[2,2,2,2,2,...,2,2,2,2,2],[2,2,2,2,1,...,2,2,2,2,2],...,[2,2,2,2,2,...,6,2,2,2,2],[2,2,2,2,2,...,2,1,2,1,2]]
tpep_pickup_datetime: [[2025-09-01 00:19:20.000000,2025-09-01 00:15:20.000000,2025-09-01 00:06:07.000000,2025-09-01 00:49:47.000000,2025-09-01 00:05:00.000000,...,2025-09-02 17:06:12.000000,2025-09-02 17:19:06.000000,2025-09-02 17:16:10.000000,2025-09-02 17:26:27.000000,2025-09-02 17:50:45.000000],[2025-09-02 17:07:11.000000,2025-09-02 17:25:59.000000,2025-09-02 17:45:05.000000,2025-09-02 17:12:0

In [2]:
display(df)

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,2,2025-09-01 00:19:20,2025-09-01 00:45:17,1.0,9.92,1.0,N,138,114,1,42.90,6.0,0.5,10.73,0.00,1.0,66.13,2.5,1.75,0.75
1,2,2025-09-01 00:15:20,2025-09-01 00:26:08,2.0,6.82,1.0,N,93,157,1,26.80,1.0,0.5,5.86,0.00,1.0,35.16,0.0,0.00,0.00
2,2,2025-09-01 00:06:07,2025-09-01 00:22:23,1.0,3.95,1.0,N,68,13,1,19.80,1.0,0.5,5.11,0.00,1.0,30.66,2.5,0.00,0.75
3,2,2025-09-01 00:49:47,2025-09-01 01:04:49,1.0,3.14,1.0,N,234,87,1,17.70,1.0,0.5,3.52,0.00,1.0,26.97,2.5,0.00,0.75
4,2,2025-09-01 00:05:00,2025-09-01 00:15:32,6.0,2.81,1.0,N,230,151,1,14.90,1.0,0.5,4.13,0.00,1.0,24.78,2.5,0.00,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12861153,2,2025-11-30 23:12:44,2025-11-30 23:43:26,NaN,10.62,NaN,None,68,169,0,33.16,0.0,0.5,0.00,6.94,1.0,44.85,NaN,NaN,0.75
12861154,1,2025-11-30 23:10:35,2025-11-30 23:28:24,NaN,6.50,NaN,None,48,116,0,22.17,0.0,0.5,0.00,0.00,1.0,26.92,NaN,NaN,0.75
12861155,2,2025-11-30 23:09:43,2025-11-30 23:36:08,NaN,8.10,NaN,None,145,152,0,-4.75,0.0,0.5,0.00,0.00,1.0,4.06,NaN,NaN,0.75
12861156,1,2025-11-30 23:29:41,2025-11-30 23:47:09,NaN,5.60,NaN,None,116,48,0,21.42,0.0,0.5,0.00,0.00,1.0,26.17,NaN,NaN,0.75


# LOAD DATA INTO DOCKER-POSTGRES DB

In [3]:
from sqlalchemy import create_engine

username = "taxi"
password = "taxi"
host = "localhost"
port = "5432"
database = "taxi_db"
conn_string = f'postgresql://{username}:{password}@{host}:{port}/{database}'

# Create the connection engine
engine = create_engine(conn_string)

In [4]:
df.head(0).to_sql(
    name='raw_sales',
    schema='raw',
    con=engine,
    if_exists='replace',
    index=False
)

0

In [5]:
import psycopg2
from io import StringIO

table = "raw_sales"
schema = "raw"

def fast_load(df, table, schema, conn_string):
    conn = psycopg2.connect(conn_string)
    cur = conn.cursor()
    buf = StringIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)
    cur.copy_expert(f"COPY {schema}.{table} FROM STDIN WITH CSV", buf)  # f-string!
    conn.commit()
    cur.close()
    conn.close()

# Call it
fast_load(df, table=table, schema=schema, conn_string=conn_string)

In [6]:
# TRMINAL CHK docker exec -it taxi_pg psql -U taxi -d taxi_db -c "SELECT COUNT(*) FROM raw.raw_sales;"
df_check_c = pd.read_sql("SELECT count(*) FROM raw.raw_sales", engine)
df_check = pd.read_sql("SELECT * FROM raw.raw_sales limit 20", engine)
display(df_check_c)
display(df_check)

,count
0,12861158


,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,2,2025-11-27 00:55:15,2025-11-27 00:59:30,None,1.15,None,None,141,263,0,-2.68,0.0,0.5,0.0,0.00,1.0,1.32,None,None,0.00
1,2,2025-11-27 00:11:42,2025-11-27 00:27:41,None,3.01,None,None,42,239,0,-4.64,0.0,0.5,0.0,0.00,1.0,-0.64,None,None,0.00
2,2,2025-11-27 00:41:35,2025-11-27 01:00:59,None,9.28,None,None,14,123,0,3.15,0.0,0.5,0.0,0.00,1.0,4.65,None,None,0.00
3,2,2025-11-27 00:49:34,2025-11-27 01:00:40,None,5.99,None,None,201,154,0,20.14,0.0,0.5,0.0,5.20,1.0,31.84,None,None,0.00
4,2,2025-11-27 00:16:46,2025-11-27 00:40:53,None,13.39,None,None,14,27,0,2.17,0.0,0.5,0.0,5.20,1.0,8.87,None,None,0.00
5,2,2025-11-27 00:30:14,2025-11-27 00:40:34,None,5.15,None,None,88,233,0,3.84,0.0,0.5,0.0,0.00,1.0,8.59,None,None,0.75
6,2,2025-11-27 00:43:47,2025-11-27 01:10:45,None,10.34,None,None,68,228,0,-13.21,0.0,0.5,0.0,6.94,1.0,2.87,None,None,0.75
7,2,2025-11-27 00:05:36,2025-11-27 00:20:29,None,2.74,None,None,231,186,0,1.09,0.0,0.5,0.0,0.00,1.0,5.84,None,None,0.75
8,1,2025-11-27 00:43:07,2025-11-27 01:23:42,None,8.60,None,None,232,152,0,41.43,0.0,0.5,0.0,0.00,1.0,46.18,None,None,0.75
9,2,2025-11-27 00:21:49,2025-11-27 00:32:43,None,2.24,None,None,90,48,0,-4.33,0.0,0.5,0.0,0.00,1.0,0.42,None,None,0.75
